In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

%matplotlib inline

In [ ]:
class SalesDataFrame:
    def __init__(self):
        self.data = None

    def load(self, filepath):
        self.data = pd.read_csv(filepath)
        print("Dataset loaded successfully!")

    def __del__(self):
        self.data = None

    def explore(self, option=1):
        if self.data is None:
            print("No dataset loaded.")
            return
        if option == 1:
            return self.data.head()
        elif option == 2:
            return self.data.tail()
        elif option == 3:
            return self.data.columns.tolist()
        elif option == 4:
            print(self.data.info())
            return self.data.dtypes

    def reset(self, other=None):
        if other is not None:
            self.data = other.data.copy() if other.data is not None else None
        else:
            self.data = None

    def mathematical(self, col):
        if self.data is None:
            print("No dataset loaded.")
            return
        return {
            "Sum": self.data[col].sum(),
            "Mean": self.data[col].mean(),
            "Median": self.data[col].median(),
            "Std": self.data[col].std(),
            "Min": self.data[col].min(),
            "Max": self.data[col].max(),
        }

    def combine(self, other, how="inner", on=None):
        if self.data is None or other.data is None:
            print("One or both datasets not loaded.")
            return None
        if on:
            return pd.merge(self.data, other.data, how=how, on=on)
        return pd.concat([self.data, other.data], ignore_index=True)

    def create_pivot(self, index, columns, values, aggfunc="sum"):
        if self.data is None:
            return None
        return pd.pivot_table(self.data, index=index, columns=columns, values=values, aggfunc=aggfunc)

    def split(self, col=None, value=None, regex=None):
        if self.data is None:
            return None, None
        if col and value is not None:
            mask = self.data[col] == value
            return self.data[mask].reset_index(drop=True), self.data[~mask].reset_index(drop=True)
        if col and regex:
            mask = self.data[col].astype(str).str.contains(regex, na=False)
            return self.data[mask].reset_index(drop=True), self.data[~mask].reset_index(drop=True)
        mid = len(self.data) // 2
        return self.data.iloc[:mid].reset_index(drop=True), self.data.iloc[mid:].reset_index(drop=True)

    def search_sort(self, col=None, value=None, ascending=True, top_n=None):
        if self.data is None:
            return None
        result = self.data.copy()
        if col and value is not None:
            result = result[result[col] == value]
        if col:
            result = result.sort_values(by=col, ascending=ascending)
        if top_n:
            result = result.head(top_n)
        return result

    def filter(self, col, condition, value):
        if self.data is None:
            return None
        ops = {
            ">": self.data[col] > value,
            "<": self.data[col] < value,
            "==": self.data[col] == value,
            ">=": self.data[col] >= value,
            "<=": self.data[col] <= value,
            "!=": self.data[col] != value,
        }
        return self.data[ops[condition]] if condition in ops else None

    def aggregate(self, col, func="sum"):
        if self.data is None:
            return None
        funcs = {
            "sum": self.data[col].sum,
            "mean": self.data[col].mean,
            "min": self.data[col].min,
            "max": self.data[col].max,
            "count": self.data[col].count,
            "median": self.data[col].median,
        }
        return funcs[func]() if func in funcs else None

    def statistical(self):
        if self.data is None:
            return None
        return self.data.describe()

    def descriptive_statistics(self):
        if self.data is None:
            print("No dataset loaded.")
            return
        display(self.data.describe(include="all"))
        print("\nSkewness:")
        display(self.data.select_dtypes(include=[np.number]).skew())
        print("\nVariance:")
        display(self.data.select_dtypes(include=[np.number]).var())
        print("\nPercentiles (25%, 50%, 75%):")
        display(self.data.select_dtypes(include=[np.number]).quantile([0.25, 0.5, 0.75]))

    def handle_missing(self, strategy="drop", col=None, fill_value=None):
        if self.data is None:
            print("No dataset loaded.")
            return
        missing = self.data.isnull().sum()
        if missing.sum() == 0:
            print("No missing values found in the dataset!")
            return
        print("Missing values per column:")
        display(missing[missing > 0])
        if strategy == "drop":
            self.data.dropna(inplace=True)
            self.data.reset_index(drop=True, inplace=True)
            print("Rows with missing values dropped.")
        elif strategy == "fill" and col and fill_value is not None:
            self.data[col].fillna(fill_value, inplace=True)
            print(f"Missing values in '{col}' filled with {fill_value}.")
        elif strategy == "replace" and col and fill_value is not None:
            self.data[col].replace(np.nan, fill_value, inplace=True)
            print(f"Missing values in '{col}' replaced with {fill_value}.")

    def visualize_bar(self, x_col, y_col):
        if self.data is None:
            return
        plt.figure(figsize=(10, 6))
        self.data.groupby(x_col)[y_col].sum().plot(kind="bar", color="steelblue")
        plt.title(f"{y_col} by {x_col}")
        plt.tight_layout()
        plt.show()

    def visualize_line(self, x_col, y_col):
        if self.data is None:
            return
        plt.figure(figsize=(10, 6))
        plt.plot(self.data[x_col], self.data[y_col], marker="o")
        plt.title(f"{y_col} over {x_col}")
        plt.tight_layout()
        plt.show()

    def visualize_scatter(self, x_col, y_col):
        if self.data is None:
            return
        plt.figure(figsize=(10, 6))
        plt.scatter(self.data[x_col], self.data[y_col], alpha=0.7, color="coral")
        plt.title(f"Scatter: {x_col} vs {y_col}")
        plt.xlabel(x_col)
        plt.ylabel(y_col)
        plt.tight_layout()
        plt.show()

    def visualize_pie(self, col):
        if self.data is None:
            return
        plt.figure(figsize=(8, 8))
        self.data[col].value_counts().plot(kind="pie", autopct="%1.1f%%", startangle=140)
        plt.title(f"Pie Chart: {col}")
        plt.tight_layout()
        plt.show()

    def visualize_histogram(self, col):
        if self.data is None:
            return
        plt.figure(figsize=(10, 6))
        self.data[col].plot(kind="hist", bins=20, color="mediumseagreen", edgecolor="black")
        plt.title(f"Histogram: {col}")
        plt.tight_layout()
        plt.show()

    def visualize_heatmap(self):
        if self.data is None:
            return
        numeric_cols = self.data.select_dtypes(include=[np.number]).columns.tolist()
        plt.figure(figsize=(12, 8))
        sns.heatmap(self.data[numeric_cols].corr(), annot=True, cmap="coolwarm", fmt=".2f")
        plt.title("Correlation Heatmap")
        plt.tight_layout()
        plt.show()

    def visualize_stack(self, x_col, y_cols):
        if self.data is None:
            return
        plt.figure(figsize=(10, 6))
        plt.stackplot(self.data[x_col], [self.data[c] for c in y_cols], labels=y_cols)
        plt.legend(loc="upper left")
        plt.title("Stack Plot")
        plt.tight_layout()
        plt.show()

    def visualize_boxplot(self, x_col, y_col):
        if self.data is None:
            return
        plt.figure(figsize=(10, 6))
        sns.boxplot(x=self.data[x_col], y=self.data[y_col])
        plt.tight_layout()
        plt.show()

    def visualize_violin(self, x_col, y_col):
        if self.data is None:
            return
        plt.figure(figsize=(10, 6))
        sns.violinplot(x=self.data[x_col], y=self.data[y_col])
        plt.tight_layout()
        plt.show()

    def multiple_plots(self):
        if self.data is None:
            return
        numeric_cols = self.data.select_dtypes(include=[np.number]).columns.tolist()[:3]
        if not numeric_cols:
            print("No numeric columns.")
            return
        fig, axes = plt.subplots(1, len(numeric_cols), figsize=(16, 5))
        if len(numeric_cols) == 1:
            axes = [axes]
        for i, col in enumerate(numeric_cols):
            axes[i].hist(self.data[col], bins=15, color="skyblue", edgecolor="black")
            axes[i].set_title(col)
        plt.tight_layout()
        plt.show()

    def save_visualization(self, filepath):
        if self.data is None:
            print("No dataset loaded.")
            return
        numeric_cols = self.data.select_dtypes(include=[np.number]).columns.tolist()
        if not numeric_cols:
            print("No numeric columns to plot.")
            return
        plt.figure(figsize=(10, 6))
        self.data[numeric_cols[0]].plot(kind="hist", bins=20, color="steelblue", edgecolor="black")
        plt.title(f"Histogram: {numeric_cols[0]}")
        plt.savefig(filepath)
        plt.close()
        print(f"Visualization saved in {filepath} successfully!")

In [ ]:
def generate_synthetic_dataset(filepath="data/sales_data.csv"):
    os.makedirs(os.path.dirname(filepath), exist_ok=True)
    np.random.seed(42)
    n = 200
    products = ["Product A", "Product B", "Product C", "Product D", "Product E"]
    regions = ["North", "South", "East", "West", "Central"]
    df = pd.DataFrame({
        "SalesID": range(1, n + 1),
        "Product": np.random.choice(products, n),
        "Region": np.random.choice(regions, n),
        "Sales": np.random.randint(100, 1000, n),
        "Year": np.random.choice([2021, 2022, 2023], n),
    })
    df.to_csv(filepath, index=False)
    print(f"Dataset generated at: {filepath}")
    return filepath

generate_synthetic_dataset()

In [ ]:
sdf = SalesDataFrame()
sdf.load("data/sales_data.csv")

In [ ]:
sdf.explore(option=1)

In [ ]:
sdf.explore(option=2)

In [ ]:
sdf.explore(option=3)

In [ ]:
sdf.explore(option=4)

In [ ]:
sdf.handle_missing()

In [ ]:
sdf.mathematical("Sales")

In [ ]:
part1, part2 = sdf.split(col="Region", value="North")
print(f"North region: {len(part1)} rows")
print(f"Other regions: {len(part2)} rows")
display(part1.head())

In [ ]:
sdf2 = SalesDataFrame()
sdf2.load("data/sales_data.csv")
combined = sdf.combine(sdf2)
print(f"Combined rows: {len(combined)}")
display(combined.head())

In [ ]:
result = sdf.search_sort(col="Sales", ascending=False, top_n=10)
display(result)

In [ ]:
high_sales = sdf.filter("Sales", ">", 700)
display(high_sales.head())

In [ ]:
print("Sum of Sales:", sdf.aggregate("Sales", "sum"))
print("Mean of Sales:", sdf.aggregate("Sales", "mean"))
print("Max of Sales:", sdf.aggregate("Sales", "max"))

In [ ]:
pivot = sdf.create_pivot(index="Region", columns="Year", values="Sales", aggfunc="sum")
display(pivot)

In [ ]:
sdf.descriptive_statistics()

In [ ]:
sdf.visualize_bar("Region", "Sales")

In [ ]:
sdf.visualize_scatter("SalesID", "Sales")

In [ ]:
sdf.visualize_histogram("Sales")

In [ ]:
sdf.visualize_pie("Region")

In [ ]:
sdf.visualize_heatmap()

In [ ]:
sdf.visualize_boxplot("Region", "Sales")

In [ ]:
sdf.visualize_violin("Region", "Sales")

In [ ]:
sdf.multiple_plots()

In [ ]:
sdf.save_visualization("scatter_plot.png")